# 07 — Quantile Local Projections (Report)

Reading-side companion to `run_quantile_lp.py`. The script fits 162
QuantReg models (6 τ × 25 h main + 6 τ × 2 h pre-trend) and dumps three
artefacts to `data/econ/`:

- `quantile_lp_results.csv` (+ `.parquet`) — main table, **consumed by NB08**
- `pretrend_results.csv` — h ∈ {−2, −1} pre-trend check
- `quantile_lp_meta.json` — shock distribution stats + run provenance

This notebook reads those files and reproduces the displays of
`07_quantile_lp.ipynb` without re-running any estimation. This report
supersedes that original exploratory notebook, which is not part of the
replication package.

**Contract with NB08.** `quantile_lp_results.csv` schema is locked:
`tau, h, beta_shock, se_shock, pval_shock, beta_interaction,
se_interaction, pval_interaction, n_obs`.

## 0. Setup & auto-execution

In [ ]:
# ── Setup ──
import sys, json, subprocess, time
from pathlib import Path
sys.path.insert(0, "..")
from config import CFG, ECON_DIR

import numpy as np
import pandas as pd

# Toggles
FORCE_RERUN = False    # if True, regenerate all outputs via the script
N_JOBS      = 1        # 1 = sequential (default); >1 = joblib loky

SCRIPT  = Path.cwd().parent / "scripts" / "run_quantile_lp.py"
OUTPUTS = {
    "main":     ECON_DIR / "quantile_lp_results.csv",
    "pretrend": ECON_DIR / "pretrend_results.csv",
    "meta":     ECON_DIR / "quantile_lp_meta.json",
}

print("Setup OK")
print(f"  FORCE_RERUN = {FORCE_RERUN}")
print(f"  N_JOBS      = {N_JOBS}")

In [ ]:
# ── Auto-execute if outputs are missing ──
missing = [t for t, p in OUTPUTS.items() if not p.exists()]
if FORCE_RERUN or missing:
    assert SCRIPT.exists(), SCRIPT
    cmd = [sys.executable, str(SCRIPT), "--n_jobs", str(N_JOBS)]
    print("Running:", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=str(Path.cwd().parent), check=True)
    print(f"\nScript done — {(time.time()-t0)/60:.2f} min")
else:
    print("All outputs present, skipping computation.")
    for t, p in OUTPUTS.items():
        print(f"  [{t}] {p.name}")

In [ ]:
# ── Load all artefacts ──
df_main = pd.read_csv(OUTPUTS["main"])
df_pre  = pd.read_csv(OUTPUTS["pretrend"]) if OUTPUTS["pretrend"].exists() else None
with open(OUTPUTS["meta"]) as f:
    meta = json.load(f)

print(f"main:     {len(df_main)} rows × {df_main.shape[1]} cols")
if df_pre is not None:
    print(f"pretrend: {len(df_pre)} rows × {df_pre.shape[1]} cols")
print(f"meta:     run at {meta['run_timestamp_utc']}  (Python {meta['python_version']})")

## 1. β(shock) by quantile and horizon

Negative β = liquidation shock depresses returns. The leverage-cycle
prediction is β(τ=0.01, h) ≪ 0 across horizons, β(τ=0.50, h) ≈ 0,
β(τ≥0.90, h) ≥ 0.

In [ ]:
print("═══ β(shock) by quantile and horizon ═══\n")
pivot = df_main.pivot_table(index="h", columns="tau", values="beta_shock")
print(pivot.round(4).to_string())

**Conclusion.** β(τ=0.01) is sharply negative on every horizon and
deepens monotonically with h until ~h=20 before plateauing; β(τ=0.50) ≈ 0;
β at the upper tails is positive. The amplification is concentrated in
the lower tail of the return distribution — the leverage-cycle signature.

## 2. Significance map of β(shock)

`***` p<0.01, `**` p<0.05, `*` p<0.10. With N≈40 580 the table is
expected to be uniformly significant; the economic question is the
magnitude (Section 4).

In [ ]:
print("═══ Significance of β(shock) ═══\n")
sig = df_main.pivot_table(index="h", columns="tau", values="pval_shock")
stars = sig.map(lambda p: "***" if p < 0.01 else ("**" if p < 0.05 else
                          ("*" if p < 0.10 else "")))
print(stars.to_string())

## 3. Interaction β(shock × OI_high) at h = 0

Tests whether the shock's effect at impact is amplified when OI is in its
rolling top quintile. A negative interaction at τ=0.01 is consistent with
the leverage cycle (deeper tail when crowding is high).

In [ ]:
print("═══ INTERACTION β(shock × OI_high) at h=0 ═══\n")
h0 = df_main[df_main["h"] == 0].sort_values("tau")
for _, r in h0.iterrows():
    sig = ("***" if r["pval_interaction"] < 0.01 else
           "**"  if r["pval_interaction"] < 0.05 else
           "")
    print(f"  τ={r['tau']:.2f}: β_interact = {r['beta_interaction']:+.4f}  "
          f"(p={r['pval_interaction']:.4f}) {sig}")

## 4. Economic significance

With N > 40 000, statistical significance is automatic. The economically
relevant question is the magnitude of the tail-deepening effect for a
*typical* liquidation shock — measured in points of hourly Q(0.01) ETH
return per unit of `shock = log(1 + L_{t-1})`.

Shock distribution stats are loaded from `quantile_lp_meta.json` —
no need to reload the 41k-row panel.

In [ ]:
print("═══ ECONOMIC SIGNIFICANCE ═══\n")
print("── Shock variable: log(1 + L_{t-1}), non-zero hours only ──")
print(f"  N total:    {meta['n_total']:,}")
print(f"  N non-zero: {meta['n_nonzero']:,} "
      f"({100 * meta['n_nonzero'] / meta['n_total']:.1f}%)")
print(f"  Mean:   {meta['shock_mean']:.2f}")
print(f"  Median: {meta['shock_median']:.2f}")
print(f"  Std:    {meta['shock_std']:.2f}")
print(f"  P95:    {meta['shock_p95']:.2f}")
print(f"  P99:    {meta['shock_p99']:.2f}")

for name, val in [("Median", meta["shock_median"]), ("P95", meta["shock_p95"])]:
    usd = np.exp(val) - 1
    print(f"\n  {name} shock = {val:.2f} → L ≈ ${usd:,.0f}/h liquidation volume")

# ── Scenario impact on Q(0.01) ──
beta01 = df_main[df_main["tau"] == 0.01].set_index("h")["beta_shock"]
beta50 = df_main[df_main["tau"] == 0.50].set_index("h")["beta_shock"]
scenario_h = (0, 6, 12, 24)

print(f"\n── Impact on Q(0.01) of hourly ETH returns (×100 = pct points) ──\n")
print(f"{'Scenario':<35} {'Δshock':>8} "
      f"{'h=0':>8} {'h=6':>8} {'h=12':>8} {'h=24':>8}")
print("─" * 78)
for scenario, delta in [
    ("Median non-zero hour", meta["shock_median"]),
    ("P95 stress episode",   meta["shock_p95"]),
    ("+1 std dev of shock",  meta["shock_std"]),
]:
    impacts = {h: beta01.get(h, np.nan) * delta for h in scenario_h}
    print(f"{scenario:<35} {delta:>8.2f} "
          f"{impacts[0]:>+8.3f} {impacts[6]:>+8.3f} "
          f"{impacts[12]:>+8.3f} {impacts[24]:>+8.3f}")

print(f"\n── Asymmetry: |β(τ=0.01)| / |β(τ=0.50)| ──")
for h in scenario_h:
    if h in beta01.index and h in beta50.index and abs(beta50[h]) > 1e-6:
        print(f"  h={h:>2}: {abs(beta01[h]) / abs(beta50[h]):.1f}x")

**Conclusion.** A median non-zero shock pulls Q(0.01) down by ~1pp at
h=12; a P95 stress episode drives it down by ~3pp. The asymmetry ratio
|β(τ=0.01)| / |β(τ=0.50)| reaches ~19× at h=0, decaying to ~6× by h=24 —
the shock is concentrated in the lower tail and persistent.

## 5. Pre-trend check (h = −1, h = −2)

The shock is `log(1 + L_{t-1})` — liquidations during hour t−1. Pre-trend
horizons use *past* returns as dependent variable:

- `h = −1`: y = r_{t-1}, contemporaneous with the liquidation hour. A
  large negative β here is *expected* (price drop ↔ liquidation trigger).
- `h = −2`: y = r_{t-2}, one hour *before* the liquidation. Near-zero β
  ⇒ no anticipation; negative β ⇒ pre-existing stress.

The ratio |β(h=0)| / |β(h=−1)| measures the incremental association
between liquidations and *subsequent* returns beyond the contemporaneous
co-movement. A ratio ≫ 1 suggests amplification; ≈ 1 suggests pure
co-occurrence; ≪ 1 suggests the dominant pattern is the reverse
(price drop triggering liquidations).

In [ ]:
if df_pre is None:
    print("pretrend_results.csv not present — re-run with --skip_pretrend off.")
else:
    print("═══ β(shock) across negative and positive horizons ═══")
    print("h=-2: return BEFORE liquidation hour")
    print("h=-1: return DURING liquidation hour (contemporaneous)")
    print("h= 0: return AFTER liquidation hour (main spec)")
    print("h= 1: cumulative return over 2 hours after\n")

    print(f"{'tau':>6} {'h=-2':>10} {'h=-1':>10} │ "
          f"{'h=0':>10} {'h=1':>10} │ {'|h=0|/|h=-1|':>14}")
    print("─" * 75)

    def _get(df, tau, h, col="beta_shock"):
        s = df.loc[(df["tau"] == tau) & (df["h"] == h), col]
        return s.iloc[0] if len(s) else np.nan

    quantiles = sorted(df_pre["tau"].unique())
    for tau in quantiles:
        b_m2 = _get(df_pre,  tau, -2)
        b_m1 = _get(df_pre,  tau, -1)
        b_0  = _get(df_main, tau,  0)
        b_1  = _get(df_main, tau,  1)
        ratio_str = (f"{abs(b_0) / abs(b_m1):.2f}x"
                     if not np.isnan(b_m1) and abs(b_m1) > 1e-6 else "—")
        print(f"{tau:>6.2f} {b_m2:>10.4f} {b_m1:>10.4f} │ "
              f"{b_0:>10.4f} {b_1:>10.4f} │ {ratio_str:>14}")

    print("\n── Pre-trend significance (p-values) ──\n")
    print(f"{'tau':>6} {'p(h=-2)':>10} {'p(h=-1)':>10} │ {'p(h=0)':>10}")
    print("─" * 45)
    for tau in quantiles:
        p_m2 = _get(df_pre,  tau, -2, "pval_shock")
        p_m1 = _get(df_pre,  tau, -1, "pval_shock")
        p_0  = _get(df_main, tau,  0, "pval_shock")
        print(f"{tau:>6.2f} {p_m2:>10.4f} {p_m1:>10.4f} │ {p_0:>10.4f}")

In [ ]:
# ── Narrative guidance at τ = 0.01 (data-driven, mirrors NB07) ──
if df_pre is not None:
    tau_main = 0.01
    b_m2 = _get(df_pre,  tau_main, -2)
    b_m1 = _get(df_pre,  tau_main, -1)
    b_0  = _get(df_main, tau_main,  0)
    b_6  = _get(df_main, tau_main,  6)

    if not np.isnan(b_m1) and not np.isnan(b_0):
        ratio = abs(b_0) / abs(b_m1) if abs(b_m1) > 1e-6 else float("inf")
        print(f"At τ = 0.01:")
        print(f"  β(h=-2) = {b_m2:+.4f}  (pre-liquidation return)")
        print(f"  β(h=-1) = {b_m1:+.4f}  (contemporaneous with liquidation)")
        print(f"  β(h= 0) = {b_0:+.4f}  (one hour after)")
        print(f"  β(h= 6) = {b_6:+.4f}  (six hours after)")
        print(f"  Ratio |β(h=0)| / |β(h=-1)| = {ratio:.2f}\n")

        print("DATA-DRIVEN INTERPRETATION:")
        if abs(b_m2) < abs(b_m1) * 0.5:
            print("  • β(h=-2) ≪ β(h=-1): the price-liquidation association")
            print("    emerges sharply at h=-1, consistent with a mechanical")
            print("    trigger (price drop → liquidation threshold breached).")
        else:
            print("  • β(h=-2) ≈ β(h=-1): price stress persists for multiple")
            print("    hours before and during liquidation episodes.")

        if ratio > 2.0:
            print("  • |β(h=0)| ≫ |β(h=-1)|: the post-liquidation return is")
            print("    substantially larger than the contemporaneous")
            print("    association — consistent with an amplification channel.")
        elif ratio > 1.2:
            print(f"  • |β(h=0)| > |β(h=-1)| (ratio {ratio:.2f}): moderate")
            print("    evidence of additional tail-deepening beyond")
            print("    contemporaneous co-movement.")
        elif ratio > 0.8:
            print(f"  • |β(h=0)| ≈ |β(h=-1)| (ratio {ratio:.2f}): contemporaneous")
            print("    and post-liquidation associations are comparable —")
            print("    consistent with a feedback loop rather than a")
            print("    one-directional causal channel.")
        else:
            print(f"  • |β(h=0)| < |β(h=-1)| (ratio {ratio:.2f}): contemporaneous")
            print("    association is STRONGER than the post-liquidation one.")
            print("    Dominant pattern is the price drop triggering")
            print("    liquidations.")

        if not np.isnan(b_6) and abs(b_6) > abs(b_0) * 1.5:
            print("  • However, |β(h=6)| ≫ |β(h=0)|: the cumulative 6-hour")
            print("    association is much stronger, suggesting that the")
            print("    liquidation episode extends the tail event over")
            print("    multiple hours.")

## 6. Interpretation guide

What to look for in the table above:

1. **β(τ=0.50) ≈ 0** ⇒ liquidations don't move the median return
   (efficient market in normal regime).
2. **β(τ=0.01) ≪ 0** ⇒ liquidations push returns deeper into the left
   tail — the *Liquidation Multiplier*.
3. **β_interaction(τ=0.01, h=0) < 0** ⇒ the multiplier is larger when OI
   is high (leverage-cycle amplification).
4. **Horizon profile** — β(τ=0.01) deepens with h before plateauing,
   indicating the shock unfolds over hours rather than a single bar.
5. **Pre-trend** (Section 5) — the comparison h=−2 / h=−1 / h=0 helps
   distinguish co-movement from amplification.

## Next steps

- **Notebook 08:** block-bootstrap CIs + cross-asset placebos
  (consumes `quantile_lp_results.csv` produced here).
- **`scripts/paper/make_figures.py`:** publication figures (IRF curves
  vs h at each τ).